# 支線診斷：每個 library 的 register 落在哪，以及 `----------` 把它拉去了哪

`DNAFunctionPredictor.energy()` 對 conv2 的輸出取 **global max**：

```python
x = self.conv2(self.conv1(dna_seq))
return torch.max(x.reshape(x.size(0), -1), dim=1, keepdim=True).values
```

conv2 是凍結的稀疏 0/1 遮罩，等於把 8 個 conv1 filter 綁成一把**間距固定的梳子**。
這把梳子在 80-mer 上只有少數幾種擺法，究竟停在哪一種，是**每條序列當場 argmax 選出來的**，
不是學來的參數，也沒有任何東西在約束它。下面把那個位置索引稱為 **register**。

register 為什麼重要：max 的反向傳播只會把梯度送到贏家那一格，所以
**register 停在哪，就決定了每個 conv1 filter 到底在學序列的哪一段**。

## 這本 notebook 要回答的

1. 每個 library 的 register 分布長什麼樣？和該 library 真正的 −35 位置對得上嗎？
2. UL 尾端那 10 個 `-` 對 register 有多少影響？

第 2 題用**反事實**回答，而不是重訓：拿**同一份權重**，只把輸入的尾端換掉。
這樣 register 的任何移動都能完全歸因於那 10 個位置，不受重訓隨機性干擾。

> 這本 notebook 是唯讀分析，不改動任何模型、權重或設計流程。

## 1. 設定與出處

`CHECKPOINT` 預設指向現行的 `weights_CorePromoter_clean.pt`。因為這個檔案會隨著重訓被覆蓋，
下面把它的大小、mtime 和 metadata 一起印出來，讓結果可以歸屬到特定一版模型。

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

import automated_promoter_library_design as r
import recursive_corepromoter_design as legacy

DEVICE = torch.device("cpu")
CHECKPOINT = legacy.WEIGHTS_DIR / "weights_CorePromoter_clean.pt"
OUT_DIR = legacy.PROJECT_ROOT / "outputs" / "register_diagnostics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 80
N_PER_LIB = 1500          # 每個 library 取樣幾條；保留訓練分布（含重複抽樣）
RNG = np.random.default_rng(777)

stat = CHECKPOINT.stat()
print("checkpoint:", CHECKPOINT.name, f"({stat.st_size} bytes, mtime_ns={stat.st_mtime_ns})")
meta_path = CHECKPOINT.with_name(CHECKPOINT.stem + "_metadata.json")
if meta_path.exists():
    print("metadata: ", json.dumps(json.loads(meta_path.read_text()), indent=2))

core_model = r.load_core_model(DEVICE, checkpoint_path=CHECKPOINT).eval()
print("loaded:", type(core_model).__name__)

## 2. 梳子的幾何

`conv2` 每個 (channel, filter) 只有一個非零位置，所以整層可以被還原成一組 **tap 偏移量**。

### 三個容易混淆的長度

| | 公式 | 值 | 意義 |
|---|---|---|---|
| **Receptive field** | `k1 + k2 − 1` | 72 | 這一疊卷積**可能**看到的輸入範圍，**含權重恆為 0 的位置**。決定 register 數 |
| 梳子跨距 | `max(tap) − min(tap) + k1` | 67 / 68 / 69 | 第一個到最後一個非零權重，隨 spacer 16/17/18 而異 |
| 實際帶權重 | `n_filters × k1` | 64 | 8 個 filter × 8 bp，彼此不重疊；三個 channel 都一樣 |

receptive field 是標準的兩層 stride-1 卷積算術：一個 conv2 輸出看 **65 個連續的** conv1 輸出，
每個 conv1 輸出看 8 個連續鹼基，而相鄰 conv1 輸出只差 1 bp ——
所以攤開來是 `65 + (8−1) = 72` bp。用長度鏈驗證會得到同一個數：

```
conv1:  80 → 80 − 8  + 1 = 73
conv2:  73 → 73 − 65 + 1 =  9  = 80 − 72 + 1   ← register 數
```

**72 bp 塞進 80 bp，只剩 8 bp 可以滑，所以只有 9 個 register。**
（對照：v0 是 `9 + 52 − 1 = 60`，有 21 個 register，滑動自由度是這裡的兩倍多。）

72 與 64 的差額來自 kernel 前緣恆為零的 3 格（最小 tap = 3），加上內部空隙：
`[19,20,21]` 永遠是空的（filter 1 與 filter 2 之間，即 UP 與 −35 之間），
再加上 spacer 造成的 `[30]`（spacer 17）或 `[30,31]`（spacer 18）。
下一格會把這些印出來 —— 跨距 67/68/69 應與 `Model_CorePromoter_clean.ipynb` 裡
`expected_logo_lengths` 斷言的值一致。

`N_REG` 一律從 `architecture_logits` 的輸出形狀取，不寫死，並與 `k1 + k2 − 1` 交叉驗證。

In [ ]:
def conv2_taps(model):
    '''conv2 是凍結的稀疏 0/1 遮罩：每個 (channel, filter) 恰好一個非零位置。'''
    w = model.conv2.weight.detach().cpu().numpy()
    taps = np.zeros(w.shape[:2], dtype=int)
    for ch in range(w.shape[0]):
        for f in range(w.shape[1]):
            nz = np.flatnonzero(np.abs(w[ch, f]) > 1e-6)
            assert len(nz) == 1, f"unexpected conv2 sparsity at ({ch}, {f}): {nz}"
            taps[ch, f] = nz[0]
    return taps


TAPS = conv2_taps(core_model)
N_CH, N_FILT = TAPS.shape
FILT_W = int(core_model.conv1.weight.shape[2])
K2 = int(core_model.conv2.weight.shape[2])
M35_SLOT = 2                      # filter slot 2 = the -35 box

with torch.no_grad():
    N_REG = int(core_model.architecture_logits(torch.zeros(1, 4, SEQ_LEN)).shape[2])

# Receptive field of stacked stride-1 convs. NOT max(tap) + FILT_W: that measures
# the span of nonzero taps, which only coincides with the receptive field when the
# outermost tap happens to sit on the kernel's last position (it does here, at
# k2-1, but only for the widest-spacer channel).
RF = FILT_W + K2 - 1
assert N_REG == SEQ_LEN - RF + 1, (N_REG, SEQ_LEN, RF)

print(f"conv1: {N_FILT} filters of width {FILT_W}    conv2: {N_CH} channels of width {K2}")
print(f"  receptive field = k1 + k2 - 1 = {FILT_W} + {K2} - 1 = {RF} bp on a {SEQ_LEN} bp input")
print(f"  conv1 out {SEQ_LEN - FILT_W + 1} -> conv2 out {N_REG} = {SEQ_LEN} - {RF} + 1 registers")
print(f"  {N_CH} channels x {N_REG} registers = {N_CH * N_REG} candidates per sequence\n")
print(f"  {N_FILT} x {FILT_W} = {N_FILT * FILT_W} bp carry weight, so {RF} - {N_FILT * FILT_W} "
      f"= {RF - N_FILT * FILT_W} bp of every receptive field are always zero.")
print("  Where those unweighted bases sit depends on the spacer channel:\n")
for ch in range(N_CH):
    lo, hi = int(TAPS[ch].min()), int(TAPS[ch].max()) + FILT_W - 1
    covered = set()
    for t in TAPS[ch]:
        covered |= set(range(int(t), int(t) + FILT_W))
    gaps = sorted(set(range(lo, hi + 1)) - covered)
    lead, trail = lo, RF - 1 - hi
    assert lead + len(gaps) + trail == RF - N_FILT * FILT_W
    print(f"  channel {ch} (spacer {legacy.SPACER_BY_CHANNEL[ch]}): taps {TAPS[ch].tolist()}")
    print(f"      comb spans kernel positions {lo}..{hi} = {hi - lo + 1} bp"
          f"   [matches expected_logo_lengths in Model_CorePromoter_clean.ipynb]")
    print(f"      unweighted: {lead} leading + {len(gaps)} internal {gaps} + {trail} trailing"
          f" = {lead + len(gaps) + trail}")

# The three channels do NOT get equal positional freedom. nn.Conv1d has ONE kernel
# width for all output channels, so the output is (N, n_ch, N_REG) no matter where a
# channel's taps sit - channel 0's kernel positions 63..64 are simply always zero.
# Sized to its own comb it would have had more placements, and the ones it loses are
# all at the 3' end.
print("\n  placement budget per channel (shared kernel vs a kernel sized to that comb):")
for ch in range(N_CH):
    k_min = int(TAPS[ch].max()) + 1
    own = (SEQ_LEN - FILT_W + 1) - k_min + 1
    reach = (N_REG - 1) + int(TAPS[ch].max()) + FILT_W - 1
    unreachable = ("reaches the 3' end" if reach == SEQ_LEN - 1
                   else f"can never read bases {reach + 1}..{SEQ_LEN - 1}")
    print(f"    channel {ch} (spacer {legacy.SPACER_BY_CHANNEL[ch]}): {N_REG} registers shared, "
          f"{own} if k2 were {k_min} (loses {own - N_REG}); {unreachable}")

## 3. 三個輸入變體（反事實的核心）

只有 **UL** 在三個 arm 之間不同，其餘六個 library 逐位元組完全相同 ——
所以任何差異都能歸因於 UL 尾端那 10 個位置。

| arm | UL 尾端 | `'-'` 的編碼 |
|---|---|---|
| `pad_minus1` | `----------` | `[-1,-1,-1,-1]` ← **現況** |
| `pad_zero` | `----------` | `[0,0,0,0]`（中性；其他四本 notebook 用這個） |
| `real_tail` | `AAATCTATGT` | 不適用（沒有 dash） |

`AAATCTATGT` 是 `Model_Dis` / `Model_ITS` / `Model_Sp17` 裡 `UL()` 一直在用的真實序列。
這裡**不修改** `legacy.dna_one_hot`，改用本地的 `one_hot(seq, dash)`。

In [ ]:
DASH_MINUS1 = (-1.0, -1.0, -1.0, -1.0)
DASH_ZERO = (0.0, 0.0, 0.0, 0.0)
UL_REAL_TAIL = "AAATCTATGT"     # what Model_Dis / Model_ITS / Model_Sp17 still use
UL_VAR = slice(9, 28)           # the variable 19 bp UP element inside the UL construct
UL_PREFIX, UL_BODY = "CAGAAAAAG", "GGCTTGCGGCTTTTGCCGCTTTTTTTTACCCTGCACACCCCT"


def one_hot(seq, dash=DASH_MINUS1):
    mapping = {
        "A": (1.0, 0.0, 0.0, 0.0), "C": (0.0, 1.0, 0.0, 0.0),
        "G": (0.0, 0.0, 1.0, 0.0), "T": (0.0, 0.0, 0.0, 1.0),
        "-": tuple(float(v) for v in dash), "N": (0.25, 0.25, 0.25, 0.25),
    }
    return np.array([mapping.get(b, (0.0, 0.0, 0.0, 0.0)) for b in str(seq).upper()],
                    dtype=np.float32).T


def UL_real(var19):
    '''The UL construct with the real 80 bp tail instead of ----------.'''
    return UL_PREFIX + var19 + UL_BODY + UL_REAL_TAIL


# the local encoder must agree with the shipped one on the current arm
np.testing.assert_allclose(one_hot("ACGTN-", DASH_MINUS1), legacy.dna_one_hot("ACGTN-"))
# and the two UL builders must differ in exactly the last 10 positions
_v = "A" * 19
assert legacy.UL(_v)[:70] == UL_real(_v)[:70]
assert legacy.UL(_v)[70:] == "-" * 10
assert UL_real(_v)[70:] == UL_REAL_TAIL and len(UL_real(_v)) == SEQ_LEN
print("encoder and constructs agree with the shipped versions")

## 4. 取樣

用 `legacy.build_core_training_dataframe()`，也就是**訓練時實際看到的那個分布**
（含 stratified sampling 的重複抽樣）。UL 的可變區可以從組裝好的序列切回來，
所以 `real_tail` arm 不需要重讀 pickle —— 切片會用 round-trip assert 驗證。

In [ ]:
df_train = legacy.build_core_training_dataframe()

samples = {}
for lib in legacy.LIBRARY_ORDER:
    seqs = df_train.loc[df_train["Library"] == lib, "Sequence"].to_numpy()
    take = RNG.choice(len(seqs), min(N_PER_LIB, len(seqs)), replace=False)
    samples[lib] = [str(s) for s in seqs[take]]
    assert all(len(s) == SEQ_LEN for s in samples[lib]), f"{lib}: not all 80 bp"

# UL's variable region round-trips, so the real-tail arm is rebuilt from it
assert all(legacy.UL(s[UL_VAR]) == s for s in samples["UL"])
samples_real = dict(samples)
samples_real["UL"] = [UL_real(s[UL_VAR]) for s in samples["UL"]]

ARMS = {
    "pad_minus1": ("UL tail ----------, '-' = [-1,-1,-1,-1]  (current)", samples, DASH_MINUS1),
    "pad_zero":   ("UL tail ----------, '-' = [0,0,0,0]", samples, DASH_ZERO),
    "real_tail":  (f"UL tail {UL_REAL_TAIL}, no dashes", samples_real, DASH_ZERO),
}
print(df_train.groupby("Library").size().reindex(legacy.LIBRARY_ORDER).to_string())
print(f"\nsampled {N_PER_LIB} per library for the diagnostics")

## 5. 抽出 register

`architecture_logits` 回傳的正是取 max **之前**的 conv2 輸出 `(N, channel, register)`，
所以 argmax 直接就是 `energy()` 實際選中的那一格。

In [ ]:
def registers(model, seqs, dash):
    '''Return (channel, register, energy) chosen by the global max, per sequence.'''
    x = torch.tensor(np.stack([one_hot(s, dash) for s in seqs]),
                     dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = model.architecture_logits(x)
    flat = logits.reshape(logits.shape[0], -1)
    best = flat.argmax(1).cpu().numpy()
    return best // N_REG, best % N_REG, flat.max(1).values.cpu().numpy()


reg_data = {}
for arm, (_, sample_set, dash) in ARMS.items():
    for lib in legacy.LIBRARY_ORDER:
        ch, reg, energy = registers(core_model, sample_set[lib], dash)
        reg_data[(arm, lib)] = {"channel": ch, "register": reg, "energy": energy}

# ---- self-check 1 -----------------------------------------------------------
# The six non-UL libraries are byte-identical across arms, so their registers must
# be identical too. If this trips, the counterfactual is wired wrong.
for lib in legacy.LIBRARY_ORDER:
    if lib == "UL":
        continue
    base = reg_data[("pad_minus1", lib)]["register"]
    for arm in ("pad_zero", "real_tail"):
        np.testing.assert_array_equal(
            reg_data[(arm, lib)]["register"], base,
            err_msg=f"{lib} moved between arms - its input should be identical")
print("self-check 1 PASS: the six non-UL libraries are unmoved across all three arms")

## 6. 圖一 — 每個 library 的 register 分布

### 橫軸是什麼

register 就是**整把梳子從 5′ 端往右平移了幾個 bp**：

```
輸入 80 bp → conv1 (kernel 8) → 73 → conv2 (kernel 65) → 9 個位置
                                              register = 0 .. 8
```

register = `r` 時，filter `f` 讀的是序列位置 `r + tap[f]` 到 `r + tap[f] + FILT_W - 1`。
以 −35 那格（filter slot 2，tap = 22）為例：`r = 7` → 讀 29..36，`r = 8` → 讀 30..37。
橫軸每往右一格，八個 filter 就**一起**往下游滑 1 bp。

register 不是學來的參數，是每條序列 forward 時當場 argmax 選出來的。

### 深淺是什麼

**每一列是一個 library，整列加總 = 100%**。格子裡的數字就是百分比，顏色只是同一個數字的
視覺編碼（單一色相，越深越高）。空白格代表低於 0.5%。

- 深且**集中在單一格** → 有東西把梳子牢牢釘住，通常就是真的 −35。
- **分散** → 沒有強力錨點。
- 停在**邊界值**（0 或 `N_REG-1`）→ 警訊：梳子被推到極限，多半是被序列端點的假訊號拉走。

### 黑框為什麼只有四個 library 有

黑框標的是「**這個構築的 −35 應該讓梳子停在哪**」，需要獨立知道 −35 六聯體的絕對位置。
能不能知道，取決於**該 library 變動的是啟動子的哪一段**：

| library | 變的是 | −35 位置可知？ |
|---|---|---|
| PL17 | −35 與 −10 六聯體本身 | ✅ 可變區起點就是 −35 |
| SL16 / SL17 / SL18 | spacer | ✅ spacer 的定義就是「−35 與 −10 之間」，緊鄰前面那 6 bp 必為 −35 |
| DL | discriminator（−10 下游） | ❌ |
| UL | UP element（上游） | ❌ |
| ITS | 轉錄起始區（TSS 下游） | ❌ |

前四個是**從 constructor 字串直接讀出來**的，不是推測（下一格會把它算出來並驗證六聯體字面值）。
後三個變的是上游或下游，−35 埋在固定前綴中間，設計本身沒有任何邊界標示它。

**為什麼不乾脆用模型讀到的位置回推？** 因為那會讓這張圖變成循環論證 ——
黑框畫在模型已經去的地方，就永遠不可能不一致。**黑框唯一的價值，就是它有可能和深色格子對不上。**
所以寧可留白。（UL 尤其不能標：那裡根本沒有像樣的 −35，模型讀到的是下游終止子髮夾的一部分。）

下一格的表格會印出每個 library 在眾數 register 上 filter2 實際讀到的 8 bp，
沒有黑框的三個可以用它自己判斷。

In [ ]:
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
# Sequential = one hue, light -> dark (documented blue ramp, steps 100 -> 700).
BLUE_STEPS = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
              "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ_CMAP = LinearSegmentedColormap.from_list("viz_blue", BLUE_STEPS)

BUILDERS = {"PL17": (legacy.PpurR, 12), "SL16": (legacy.SL16, 16), "SL17": (legacy.SL17, 17),
            "SL18": (legacy.SL18, 18), "DL": (legacy.DL, 8), "UL": (legacy.UL, 19),
            "ITS": (legacy.ITS_context, 10)}
VARIES = {"PL17": "-35 and -10 hexamers", "SL16": "spacer", "SL17": "spacer", "SL18": "spacer",
          "DL": "discriminator (downstream of -10)", "UL": "UP element (upstream)",
          "ITS": "initial transcribed region (downstream of TSS)"}
M35_LEN = 6


def variable_blocks(fn, n_var):
    '''Which positions of the assembled 80-mer come from the library's variable region.'''
    probe = fn("N" * n_var)
    idx = [i for i, c in enumerate(probe) if c == "N"]
    blocks, start = [], idx[0]
    for a, b in zip(idx, idx[1:] + [None]):
        if b != a + 1:
            blocks.append((start, a))
            start = b
    return blocks


BLOCKS = {lib: variable_blocks(*BUILDERS[lib]) for lib in legacy.LIBRARY_ORDER}

# The -35 start is readable off the construct ONLY where the design puts it on a
# boundary: PL17 varies the hexamer itself, and the SL libraries vary the spacer,
# which by definition begins right after the -35. Deriving it (rather than hard-
# coding) means the rule is visible, and the assert below checks it really landed
# on a -35-like hexamer. DL / UL / ITS vary something up- or downstream, so the
# construct says nothing about where their -35 is - they stay unannotated rather
# than being back-inferred from the model, which would make the marker circular.
M35_ABS = {"PL17": BLOCKS["PL17"][0][0]}
for lib in ("SL16", "SL17", "SL18"):
    M35_ABS[lib] = BLOCKS[lib][0][0] - M35_LEN

_probe = {lib: BUILDERS[lib][0]("N" * BUILDERS[lib][1]) for lib in legacy.LIBRARY_ORDER}
_hex = {lib: _probe[lib][p:p + M35_LEN] for lib, p in M35_ABS.items() if lib != "PL17"}
assert _hex == {"SL16": "TTGACT", "SL17": "TTGACC", "SL18": "TTTACA"}, _hex
print("derived -35 starts:", M35_ABS, "| spacer-library hexamers:", _hex)

EXPECTED_REG = {lib: pos - int(TAPS[0, M35_SLOT]) for lib, pos in M35_ABS.items()}


def mode_of(a):
    v, c = np.unique(a, return_counts=True)
    return int(v[c.argmax()]), float(c.max() / len(a))


rows = []
for lib in legacy.LIBRARY_ORDER:
    d = reg_data[("pad_minus1", lib)]
    m_reg, share = mode_of(d["register"])
    m_ch, _ = mode_of(d["channel"])
    start = m_reg + int(TAPS[m_ch, M35_SLOT])
    rows.append({
        "library": lib,
        "varies": VARIES[lib],
        "variable_region": ", ".join(f"{a}-{b}" for a, b in BLOCKS[lib]),
        "mode_register": m_reg,
        "mode_share": share,
        "mean_register": float(d["register"].mean()),
        "expected_register": EXPECTED_REG.get(lib, np.nan),
        # how many registers to spare before the -35 becomes unreachable
        "headroom": (N_REG - 1 - EXPECTED_REG[lib]) if lib in EXPECTED_REG else np.nan,
        "spacer": legacy.SPACER_BY_CHANNEL[m_ch],
        "mean_energy": float(d["energy"].mean()),
        "filter2_window": f"{start}:{start + FILT_W}",
        "filter2_reads": ARMS["pad_minus1"][1][lib][0][start:start + FILT_W],
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "register_summary.csv", index=False)
display(summary.round(3))

In [ ]:
share = np.zeros((len(legacy.LIBRARY_ORDER), N_REG))
for i, lib in enumerate(legacy.LIBRARY_ORDER):
    reg = reg_data[("pad_minus1", lib)]["register"]
    share[i] = np.bincount(reg, minlength=N_REG) / len(reg)

fig, ax = plt.subplots(figsize=(10.4, 4.8), dpi=140)
im = ax.imshow(share, cmap=SEQ_CMAP, vmin=0.0, vmax=1.0, aspect="auto")

# Table view of the same numbers: label every non-empty cell, so nothing is
# carried by color alone. Empty cells stay blank rather than printing "0%".
for i in range(share.shape[0]):
    for j in range(N_REG):
        if share[i, j] < 0.005:
            continue
        ax.text(j, i, f"{share[i, j] * 100:.0f}", ha="center", va="center", fontsize=8.5,
                color="#ffffff" if share[i, j] > 0.55 else INK)

# Expected register, only where the -35 position is fixed by design.
for i, lib in enumerate(legacy.LIBRARY_ORDER):
    if lib in EXPECTED_REG:
        ax.add_patch(plt.Rectangle((EXPECTED_REG[lib] - 0.5, i - 0.5), 1, 1,
                                   fill=False, edgecolor=INK, lw=1.8, zorder=3))

ax.set_xticks(range(N_REG))
ax.set_yticks(range(len(legacy.LIBRARY_ORDER)))
ax.set_yticklabels(legacy.LIBRARY_ORDER)
ax.set_xlabel(
    f"register  =  how many bp the whole comb is shifted from the 5' end\n"
    f"filter f then reads positions  register + tap[f] .. + {FILT_W - 1}"
    f"   (the -35 slot has tap {int(TAPS[0, M35_SLOT])}, so register 7 -> it reads 29..36)",
    fontsize=9, color=INK_MUTED, labelpad=9)
ax.tick_params(labelsize=8.5, colors=INK_MUTED, length=0)
for s in ax.spines.values():
    s.set_color(GRID)
ax.set_title(
    "Where the filter comb lands, per library\n"
    "each row sums to 100% of that library's sequences; blank = under 0.5%\n"
    "black outline = the register this construct's -35 implies - drawn only for the four "
    "libraries whose design pins it\n"
    "(no outline means the -35 position is not independently known, NOT that the model is wrong)",
    fontsize=9.5, color=INK, pad=10)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.015)
cb.set_label("share of sequences", fontsize=8.5, color=INK_MUTED)
cb.ax.tick_params(labelsize=8, colors=INK_MUTED)
cb.outline.set_edgecolor(GRID)
fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUT_DIR / f"register_distribution.{ext}", bbox_inches="tight", facecolor=SURFACE)
plt.show()

## 7. 圖二 — UL 的 padding 反事實

同一份權重，只換 UL 尾端那 10 個位置。這一格如果顯示三條分布**明顯不同**，
就代表 register 是被 padding 決定的，而不是被 UP element 或任何啟動子訊號決定的。

配色用經過驗證的前三個 categorical slot（all-pairs，light mode：最差 CVD ΔE 9.2、
normal-vision ΔE 24.0，均通過）。其中 aqua 對淺色底只有 2.74:1，低於 3:1，
因此依規定補上**直接數值標籤**與下方的**表格檢視**，顏色不是唯一的辨識管道。

In [ ]:
# Categorical slots 1-3; validated all-pairs, light mode (see notes above).
ARM_COLOR = {"pad_minus1": "#2a78d6", "pad_zero": "#eb6834", "real_tail": "#1baf7a"}

ul_share = {arm: np.bincount(reg_data[(arm, "UL")]["register"], minlength=N_REG)
            / len(reg_data[(arm, "UL")]["register"]) for arm in ARMS}

ul_rows = []
for arm, (label, _, _) in ARMS.items():
    d = reg_data[(arm, "UL")]
    m_reg, sh = mode_of(d["register"])
    ul_rows.append({"arm": arm, "input": label, "mode_register": m_reg,
                    "mode_share": sh, "mean_register": float(d["register"].mean()),
                    "mean_energy": float(d["energy"].mean())})
ul_summary = pd.DataFrame(ul_rows)
ul_summary.to_csv(OUT_DIR / "ul_padding_counterfactual.csv", index=False)
display(ul_summary.round(3))

x = np.arange(N_REG)
width = 0.26
fig, ax = plt.subplots(figsize=(10.5, 4.0), dpi=140)
for k, (arm, (label, _, _)) in enumerate(ARMS.items()):
    off = (k - 1) * (width + 0.02)          # 2px-equivalent surface gap between bars
    bars = ax.bar(x + off, ul_share[arm], width, color=ARM_COLOR[arm],
                  label=f"{arm} - {label}", zorder=3)
    for xi, v in zip(x + off, ul_share[arm]):
        if v >= 0.02:                        # selective direct labels, not one per bar
            ax.text(xi, v + 0.02, f"{v * 100:.0f}", ha="center", va="bottom",
                    fontsize=8, color=INK)

ax.set_xticks(x)
ax.set_xlabel("register", fontsize=9.5, color=INK_MUTED)
ax.set_ylabel("share of UL sequences", fontsize=9.5, color=INK_MUTED)
ax.set_ylim(0, 1.12)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.tick_params(labelsize=8.5, colors=INK_MUTED)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(GRID)
ax.grid(axis="y", color=GRID, lw=0.6, zorder=0)
ax.legend(fontsize=8.5, frameon=False, loc="upper left", labelcolor=INK_MUTED)
ax.set_title("Same weights, three UL tails: where does the comb park?",
             fontsize=12, color=INK, pad=10)
fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUT_DIR / f"ul_padding_counterfactual.{ext}", bbox_inches="tight",
                facecolor=SURFACE)
plt.show()

### 7b. 梳子停的那一格，是不是「padding 最多」的那一格？

上面的分布顯示 register 被 padding 帶著走。這裡問一個更尖銳的問題：
在全部 `N_CH × N_REG` 個候選位置裡，模型替 UL 選的那一格，
和「dash 欄位覆蓋最多」的那一格是不是同一個？

注意三個 channel 的機會**並不對等** —— 因為 `nn.Conv1d` 只有一個共用的 kernel 寬度，
tap 較短的 channel 構不到序列 3′ 端（見 §2 的 placement budget），
所以它們能吃到的 padding 天生就比較少。

In [ ]:
ul_probe = legacy.UL("N" * 19)
dash_pos = {i for i, b in enumerate(ul_probe) if b == "-"}

cover = np.zeros((N_CH, N_REG), dtype=int)
for ch in range(N_CH):
    for reg in range(N_REG):
        cover[ch, reg] = sum(1 for f in range(N_FILT) for k in range(FILT_W)
                             if (reg + int(TAPS[ch, f]) + k) in dash_pos)

display(pd.DataFrame(
    cover,
    index=[f"ch{c} (spacer {legacy.SPACER_BY_CHANNEL[c]})" for c in range(N_CH)],
    columns=[f"r{rr}" for rr in range(N_REG)]))

best_ch, best_reg = (int(v) for v in np.unravel_index(cover.argmax(), cover.shape))
chosen_ch, _ = mode_of(reg_data[("pad_minus1", "UL")]["channel"])
chosen_reg, _ = mode_of(reg_data[("pad_minus1", "UL")]["register"])

print(f"UL has {len(dash_pos)} dash positions ({min(dash_pos)}..{max(dash_pos)})")
print(f"most padding under the comb: channel {best_ch} "
      f"(spacer {legacy.SPACER_BY_CHANNEL[best_ch]}), register {best_reg} "
      f"-> {cover[best_ch, best_reg]} dash columns")
print(f"what the model actually picks:  channel {chosen_ch} "
      f"(spacer {legacy.SPACER_BY_CHANNEL[chosen_ch]}), register {chosen_reg}")

# ---- self-check 4 -----------------------------------------------------------
assert (best_ch, best_reg) == (chosen_ch, chosen_reg), (
    f"UL's placement {(chosen_ch, chosen_reg)} is no longer the padding argmax "
    f"{(best_ch, best_reg)} - the conclusion below needs rewriting")
print("\nself-check 4 PASS: UL's chosen placement IS the global argmax of padding coverage.")
print("The spacer-18 channel is therefore not selected because UL has an 18 bp spacer -")
print("it is selected because it is the one channel whose comb reaches the padded 3' end.")

## 8. 圖三 — 每個 filter 的能量貢獻

register 告訴你「梳子停在哪」，這裡回答「**為什麼**停在那」。
在每個 library 的眾數 (channel, register) 上，把總能量拆成 8 個 filter 各自的貢獻。

貢獻有正負、極性有意義（正 = 把梳子往這裡拉，負 = 在扣分），所以用
**diverging：兩個對立色相 + 中性灰 0 點**。

In [ ]:
def contributions(model, seqs, dash, channel, register):
    '''Per-filter energy contribution at one fixed (channel, register).'''
    x = torch.tensor(np.stack([one_hot(s, dash) for s in seqs]),
                     dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        c1 = model.conv1(x)
    return np.stack([c1[:, f, register + int(TAPS[channel, f])].cpu().numpy()
                     for f in range(N_FILT)], axis=1)


contrib = {}
modal = {}
for lib in legacy.LIBRARY_ORDER:
    d = reg_data[("pad_minus1", lib)]
    m_reg, _ = mode_of(d["register"])
    m_ch, _ = mode_of(d["channel"])
    modal[lib] = (m_ch, m_reg)
    sel = (d["register"] == m_reg) & (d["channel"] == m_ch)
    seqs = [s for s, keep in zip(ARMS["pad_minus1"][1][lib], sel) if keep]
    contrib[lib] = contributions(core_model, seqs, DASH_MINUS1, m_ch, m_reg).mean(0)

contrib_df = pd.DataFrame(contrib, index=[f"f{f}" for f in range(N_FILT)]).T
contrib_df = contrib_df.reindex(legacy.LIBRARY_ORDER)
contrib_df.to_csv(OUT_DIR / "filter_contributions.csv")
display(contrib_df.round(2))

In [ ]:
# ---- self-checks 2 and 3 ----------------------------------------------------
# The model is affine in the one-hot input, so the dash contribution is exactly
# contrib(dash=-1) - contrib(dash=0) evaluated at the SAME register.
w1 = core_model.conv1.weight.detach().cpu().numpy()      # (filter, base, col)
ul_ch, ul_reg = modal["UL"]
ul_seq0 = ARMS["pad_minus1"][1]["UL"][0]
dash_cols = {i for i, b in enumerate(ul_seq0) if b == "-"}

sel = ((reg_data[("pad_minus1", "UL")]["register"] == ul_reg)
       & (reg_data[("pad_minus1", "UL")]["channel"] == ul_ch))
ul_seqs = [s for s, keep in zip(ARMS["pad_minus1"][1]["UL"], sel) if keep]

c_minus1 = contributions(core_model, ul_seqs, DASH_MINUS1, ul_ch, ul_reg).mean(0)
c_zero = contributions(core_model, ul_seqs, DASH_ZERO, ul_ch, ul_reg).mean(0)
dash_part = c_minus1 - c_zero

# analytic expectation: each '-' column contributes -(sum of that column's weights)
expected, n_dash = np.zeros(N_FILT), np.zeros(N_FILT, dtype=int)
for f in range(N_FILT):
    start = ul_reg + int(TAPS[ul_ch, f])
    cols = [k for k in range(FILT_W) if (start + k) in dash_cols]
    n_dash[f] = len(cols)
    expected[f] = -sum(w1[f, :, k].sum() for k in cols)

np.testing.assert_allclose(dash_part, expected, atol=1e-4)
print("self-check 2 PASS: measured dash contribution matches -sum(weights) analytically")

# Under the neutral encoding a filter sitting entirely on dashes must score exactly 0.
full = np.flatnonzero(n_dash == FILT_W)
if len(full):
    np.testing.assert_allclose(c_zero[full], 0.0, atol=1e-6)
    print(f"self-check 3 PASS: filter(s) {full.tolist()} score exactly 0 under '-' = [0,0,0,0]")
else:
    print("self-check 3 skipped: no filter sits entirely on dashes at this register")

print(f"\nUL modal (channel, register) = ({ul_ch}, {ul_reg}); dash columns per filter: {n_dash.tolist()}")
print(f"total energy {c_minus1.sum():+.2f} = real bases {c_zero.sum():+.2f} + padding {dash_part.sum():+.2f}")

In [ ]:
# Diverging: two opposite hues + a neutral gray midpoint, equal steps per arm.
DIV_CMAP = LinearSegmentedColormap.from_list("viz_div", ["#2a78d6", "#f0efec", "#e34948"])
M = contrib_df.to_numpy()
lim = float(np.abs(M).max())

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.3), dpi=140,
                         gridspec_kw={"width_ratios": [2.6, 1.0], "wspace": 0.28})

ax = axes[0]
im = ax.imshow(M, cmap=DIV_CMAP, norm=TwoSlopeNorm(vmin=-lim, vcenter=0.0, vmax=lim),
               aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:+.2f}", ha="center", va="center", fontsize=8,
                color="#ffffff" if abs(M[i, j]) > 0.62 * lim else INK)
ax.set_xticks(range(N_FILT))
ax.set_xticklabels([f"f{f}" for f in range(N_FILT)])
ax.set_yticks(range(len(legacy.LIBRARY_ORDER)))
ax.set_yticklabels(legacy.LIBRARY_ORDER)
ax.tick_params(labelsize=8.5, colors=INK_MUTED, length=0)
for s in ax.spines.values():
    s.set_color(GRID)
ax.set_xlabel("conv1 filter slot (f2 = the -35 box)", fontsize=9, color=INK_MUTED)
ax.set_title("Per-filter energy contribution at each library's modal register",
             fontsize=11, color=INK, pad=10)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.015)
cb.set_label("energy", fontsize=8.5, color=INK_MUTED)
cb.ax.tick_params(labelsize=8, colors=INK_MUTED)
cb.outline.set_edgecolor(GRID)

# UL only: how the total energy is composed. The two parts have OPPOSITE signs,
# so they cannot stack - this is a waterfall, each step starting where the last ended.
ax = axes[1]
real, pad = float(c_zero.sum()), float(dash_part.sum())
total = real + pad
steps = [("real bases", 0.0, real, "#2a78d6"),
         ("'-' padding", real, total, "#eb6834"),
         ("total", 0.0, total, "#898781")]
span = max(abs(real), abs(total), abs(pad))

for xi, (name, lo, hi, colour) in enumerate(steps):
    ax.bar(xi, hi - lo, 0.55, bottom=lo, color=colour, zorder=3)
    # Label inside only when the bar is tall enough to hold it with padding;
    # otherwise place it clear of the bar end so it can never be clipped.
    if abs(hi - lo) > 0.18 * span:
        ax.text(xi, (lo + hi) / 2, f"{hi - lo:+.2f}", ha="center", va="center",
                fontsize=9.5, color="#ffffff")
    else:
        ax.text(xi, min(lo, hi) - 0.06 * span, f"{hi - lo:+.2f}", ha="center", va="top",
                fontsize=9.5, color=INK)
# hairline connectors: each step starts at the running total of the previous one
for xi in (0, 1):
    ax.plot([xi + 0.275, xi + 0.725], [steps[xi][2]] * 2, color=INK_MUTED, lw=0.8, zorder=2)

ax.axhline(0, color=INK_MUTED, lw=1.0, zorder=2)
ax.set_xticks(range(len(steps)))
ax.set_xticklabels([s[0] for s in steps], fontsize=9)
ax.set_ylim(min(real, total) - 0.22 * span, 0.16 * span)
ax.set_ylabel("energy at the modal register", fontsize=9, color=INK_MUTED)
ax.tick_params(labelsize=8.5, colors=INK_MUTED)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(GRID)
ax.grid(axis="y", color=GRID, lw=0.6, zorder=0)
ax.set_title("UL: what the energy is made of", fontsize=11, color=INK, pad=10)

fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUT_DIR / f"filter_contributions.{ext}", bbox_inches="tight", facecolor=SURFACE)
plt.show()
print("written to:", OUT_DIR)

## 怎麼讀這三張圖

**圖一（register 分布）**

- 有黑框的格子代表「該 library 的 −35 應該讓梳子停在這裡」。**顏色深的格子落在黑框上 = 對齊正確。**
- 某個 library 的機率**集中在單一 register**（接近 100%），代表有東西把梳子牢牢釘住 ——
  通常就是真的 −35。**分散**則代表沒有東西釘得住它。
- register 停在**邊界值**（0 或 `N_REG-1`）是警訊：梳子被推到極限，多半是被序列端點的
  某個假訊號拉過去，而不是被啟動子拉過去。

**圖二（UL 反事實）**

三條分布如果幾乎重合，padding 就無關緊要。如果 `pad_minus1` 明顯偏離另外兩條，
那 10 個 `-` 就在主導 UL 的 register —— 而 UL 是**唯一**把可變序列放在 filter 0/1 底下的
library，所以那兩個 filter 學到什麼，等於是被 padding 決定的。

`pad_zero` 和 `real_tail` 的差別也值得看：中性編碼讓那 10 格恆為 0，
但「恆為 0」仍然可能比一般沒對上的位置（通常是負值）更好賺，所以 `pad_zero` 未必等同
`real_tail`。這正是「該改編碼還是改序列」的判斷依據。

**圖三（filter 貢獻）**

- f2 是 −35 那一格。它在某個 library **是強烈正值** → 梳子是被真的 −35 錨定的。
  **是負值** → 那個 library 的 register 是別的東西決定的，f2 只是被拖著走。
- 右邊的堆疊長條把 UL 的總能量拆成「真實鹼基」與「padding」。橘色那段如果佔比很大，
  就是這個模型在拿 padding 換能量的直接證據。

## 已知限制

- 只適用 clean 的架構（`N_FILT` / `N_REG` 都從模型讀出，但 `M35_ABS` 的座標假設 80 bp 構築）。
  `Model_CorePromoter_v0.ipynb` 是 6 filters / conv2 kernel 52 → 21 個 register，
  而且 `weights/` 裡沒有 v0 的 checkpoint，不在範圍內。
- `expected_register` 只標 PL17 / SL16 / SL17 / SL18 四個。DL / UL / ITS 構築裡的 −35
  位置未經確認，刻意留空。
- register 是每條序列的 argmax，不是學來的參數。「平均落點」只是分布的摘要，
  **圖一每一列的完整分布才是全部資訊** —— 雙峰分布的平均值會落在兩峰之間，那裡可能一條序列都沒有。
- 反事實只換輸入、不重訓。它回答「這份權重對 padding 的反應是什麼」，
  不回答「拿掉 padding 重訓會學成什麼」—— 後者要真的重訓再跑一次這本 notebook 比對。